# Lab 06: Cloud Scheduling with Reinforcement Learning

## **Some instructions before getting started**:
<div style="font-family: 'Arial'; font-size: 16px; line-height: 1.6; text-align: justify;">

Start the Kernel: At the top right, choose <strong>Select Kernel ➞ Python Environments...</strong>

You can run all code blocks to check: From the menu bar, choose <strong>Run All</strong>.

Complete all code blocks marked with the comment <span style="font-family: monospace; font-weight: bold; color:white; background-color: green;"> ### YOU NEED TO WRITE YOUR CODE BELOW ### </span>

This notebook covers:
- **CloudSchedulingEnv** for **resource provisioning and task scheduling**
- **Student TODO sections** for RL agents, training loops, and evaluation
- **Baseline testing** for scheduling decisions and performance analysis

</div>

### Imports and Setup

The following cell configures the runtime and defines shared utilities used across all parts of this lab.

In [ ]:
import copy
import os
import random
from collections import deque

import gymnasium as gym
import numpy as np
from gymnasium import spaces

## Part 1: Cloud Scheduling Environment and Problem Setup

### 1.1. Task Model and Environment Goals

Each line in the workload file is converted into one independent task with CPU, RAM, and disk requirements.
The environment keeps track of the ready queue, running tasks, completed tasks, and rejected tasks.
Students should first understand how the environment represents workload items before attempting to modify the scheduling logic.

The main goal of this section is to show how cloud resource provisioning can be expressed as a reinforcement learning problem.

### 1.2. Environment API, State Vector, and Scheduling Actions

`CloudSchedulingEnv` follows the Gymnasium API by exposing `reset()` and `step()` methods.
The observation vector combines summary statistics, the current task description, and the flattened server resource state.
Each action selects a server and VM pair, while the wait action lets the scheduler defer the decision for one time step.
The reward encourages valid placements, discourages waiting, and penalizes invalid scheduling decisions.

In [ ]:
"""
Simple cloud scheduling environment with independent tasks, multiple servers and VMs, 
and a simple reward function based on task completion and deadlines.
"""

class CloudTask(object):
    """Represent a single independent task."""

    def __init__(self, job_id, task_index, cpu_demand, ram_demand, disk_demand, order_index):
        self.job_id = str(job_id)
        self.task_index = int(float(task_index))
        self.cpu_demand = float(cpu_demand)
        self.ram_demand = float(ram_demand)
        self.disk_demand = float(disk_demand)
        self.order_index = int(order_index)

        # Derive a synthetic runtime and deadline from the task size.
        # task runtime is proportional to the sum of resource demands, scaled by 20.0, with a minimum of 1.0.
        # 20.0 is an arbitrary scaling factor to make runtimes more realistic for the given resource demands.
        self.runtime = max(1.0, (self.cpu_demand + self.ram_demand + self.disk_demand) * 20.0)
        # task deadline is set to 4 times the runtime plus an additional 10.0 time units, to allow for some scheduling flexibility.
        self.deadline = self.runtime * 4.0 + 10.0

        self.status = "pending"
        self.ready_time = None
        self.start_time = None
        self.finish_time = None
        self.assigned_location = None

In [ ]:

class CloudSchedulingEnv(gym.Env):
    """Gymnasium-style environment for scheduling independent tasks on servers and VMs.

    Action semantics:
    - ``0`` to ``num_servers * num_vms_per_server - 1`` selects a server and VM pair.
    - ``wait_action`` means do nothing for one time step.
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, scale="small", workload_path="dataset.txt", num_tasks=None,
                 num_servers=300, num_vms_per_server=5):
        super().__init__()
        self.scale = scale
        self.workload_path = workload_path
        self.num_tasks = num_tasks
        self.num_servers = int(num_servers)
        self.num_vms_per_server = int(num_vms_per_server)

        # map flat action indices to (server_index, vm_index) pairs
        self.action_map = self._build_action_map()
        # the action space includes all server-VM pairs plus one additional "wait" action
        self.action_space_size = len(self.action_map) + 1
        # the last action index is reserved for the "wait" action
        self.wait_action = self.action_space_size - 1

        # job templates loaded from the workload file, each representing an independent task
        self.job_templates = self._load_workload(self.workload_path, self.num_tasks)
        self.total_tasks = len(self.job_templates)
        self.max_steps = max(1, self.total_tasks * 4)
        self.max_runtime = max((task.runtime for task in self.job_templates), default=1.0)
        self.max_deadline = max((task.deadline for task in self.job_templates), default=1.0)
        
        # state size = summary features + current task features + resource features for all server-VM pairs
        self.state_size = self._calculate_state_size()
        self.action_space = spaces.Discrete(self.action_space_size)
        self.observation_space = spaces.Box(
            low=0.0,
            high=np.finfo(np.float32).max,
            shape=(self.state_size,),
            dtype=np.float32,
        )

        self.reset()

    def _build_action_map(self):
        """Flatten server and VM indices into a single action list."""
        action_map = []
        for server_index in range(self.num_servers):
            for vm_index in range(self.num_vms_per_server):
                action_map.append((server_index, vm_index))
        return action_map

    def _calculate_state_size(self):
        """Return the length of the observation vector."""
        summary_features = 3
        task_features = 6
        resource_features = len(self.action_map) * 3
        return summary_features + task_features + resource_features

    def _load_workload(self, workload_path, max_tasks=None):
        """Load workload data and convert every line into one independent task."""
        if not workload_path or not os.path.exists(workload_path):
            raise FileNotFoundError("Workload file not found: {0}".format(workload_path))

        tasks = []
        order_index = 0

        with open(workload_path, "r") as file_obj:
            for raw_line in file_obj:
                line = raw_line.strip()
                if not line or line.startswith("#") or line.startswith("//"):
                    continue

                tokens = line.split()
                if len(tokens) < 7:
                    continue

                task = CloudTask(
                    job_id=tokens[1],
                    task_index=tokens[2],
                    cpu_demand=tokens[4],
                    ram_demand=tokens[5],
                    disk_demand=tokens[6],
                    order_index=order_index,
                )
                tasks.append(task)
                order_index += 1

                if max_tasks is not None and len(tasks) >= int(max_tasks):
                    break

        return tasks

    def _build_server_resources(self):
        """Create the resource matrix for all servers and VMs."""
        # For simplicity, we assume each VM has the same capacity, and the total capacity of a server is normalized to 1.0 for each resource type.
        # There are 3 resource types: CPU, RAM, and Disk. Each VM gets an equal share of the server's total capacity.
        vm_capacity = 1.0 / self.num_vms_per_server
        return [
            [[vm_capacity, vm_capacity, vm_capacity] for _ in range(self.num_vms_per_server)]
            for _ in range(self.num_servers)
        ]

    def _build_running_task_slots(self):
        """Create the running-task list for every VM."""
        return [[[] for _ in range(self.num_vms_per_server)] for _ in range(self.num_servers)]

    def _decode_action(self, action):
        """Map a flat action index back to server and VM indices."""
        if action is None:
            return None

        action_index = int(action)
        if action_index == self.wait_action:
            return None
        if action_index < 0 or action_index >= len(self.action_map):
            return None
        return self.action_map[action_index]

    def _can_place_task(self, task, server_index, vm_index):
        """Check whether the selected VM can run the current task."""
        if server_index is None:
            return False
        if server_index < 0 or server_index >= self.num_servers:
            return False
        if vm_index < 0 or vm_index >= self.num_vms_per_server:
            return False

        vm_resources = self.resources[server_index][vm_index]
        # Check if the VM has enough resources to run the task and if it can meet the task's deadline.
        has_capacity = (
            vm_resources[0] >= task.cpu_demand and
            vm_resources[1] >= task.ram_demand and
            vm_resources[2] >= task.disk_demand
        )
        # Check if the task can finish before its deadline if started now.
        will_meet_deadline = self.current_time + task.runtime <= task.deadline
        return has_capacity and will_meet_deadline

    def _allocate_task(self, task, server_index, vm_index):
        """Reserve resources and mark the task as running."""
        vm_resources = self.resources[server_index][vm_index]
        # If the task is allocated to this VM, we need to reduce the available resources accordingly.
        vm_resources[0] -= task.cpu_demand
        vm_resources[1] -= task.ram_demand
        vm_resources[2] -= task.disk_demand

        task.status = "running"
        task.start_time = self.current_time
        task.finish_time = self.current_time + task.runtime
        task.assigned_location = (server_index, vm_index)
        self.running_tasks[server_index][vm_index].append(task)

    def _release_completed_tasks(self):
        """Release tasks whose finish time has passed."""
        for server_index in range(self.num_servers):
            for vm_index in range(self.num_vms_per_server):
                still_running = []
                for task in self.running_tasks[server_index][vm_index]:
                    if task.finish_time is not None and task.finish_time <= self.current_time:
                        # If the task has finished, mark it as completed.
                        task.status = "finished"
                        self.completed_tasks += 1
                        # And, release the resources back to the VM.
                        self.resources[server_index][vm_index][0] += task.cpu_demand
                        self.resources[server_index][vm_index][1] += task.ram_demand
                        self.resources[server_index][vm_index][2] += task.disk_demand
                    else:
                        still_running.append(task)
                self.running_tasks[server_index][vm_index] = still_running

    def _task_wait_time(self, task):
        """Return how long the task has waited in the queue."""
        if task.ready_time is None:
            return 0.0
        return max(0.0, self.current_time - task.ready_time)

    def _calculate_reward(self, task, success, reason=None):
        """Compute a simple reward for scheduling success or failure."""
        ### YOU NEED TO WRITE YOUR CODE BELOW ###
        pass

    def _current_task(self):
        """Return the next task to be scheduled."""
        return self.ready_queue[0] if self.ready_queue else None

    def _finalize_ready_task(self, action):
        """Handle the current task using the chosen action."""
        task = self._current_task()
        if task is None:
            return 0.0, "idle", None, None

        if action is None or int(action) == self.wait_action:
            # Waiting scenario
            reward = self._calculate_reward(task, False, reason="wait")
            return reward, "wait", task, None

        decoded_action = self._decode_action(action)
        if decoded_action is None:
            self.ready_queue.popleft()
            self._reject_task(task)
            self.rejected_tasks += 1
            return self._calculate_reward(task, False), "rejected", task, "invalid_action"

        server_index, vm_index = decoded_action
        # Reject scenario
        if not self._can_place_task(task, server_index, vm_index):
            self.ready_queue.popleft()
            self._reject_task(task)
            self.rejected_tasks += 1
            return self._calculate_reward(task, False), "rejected", task, "resource_or_deadline_violation"

        # Schedule scenario
        self.ready_queue.popleft()
        self._allocate_task(task, server_index, vm_index)
        reward = self._calculate_reward(task, True)
        return reward, "scheduled", task, None

    def _reject_task(self, task):
        """Reject a task and mark it as not schedulable anymore."""
        task.status = "rejected"
        task.finish_time = self.current_time
        task.assigned_location = None

    def _advance_time(self):
        """Move the simulation forward by one time unit."""
        self.current_time += 1.0
        self._release_completed_tasks()

    def _count_running_tasks(self):
        """Count the number of currently running tasks."""
        count = 0
        for server_index in range(self.num_servers):
            for vm_index in range(self.num_vms_per_server):
                count += len(self.running_tasks[server_index][vm_index])
        return count

    def _is_done(self):
        """Check whether the episode is finished."""
        # The episode is done if there are no ready tasks, no running tasks, and all tasks have been handled (either completed or rejected).
        no_ready_tasks = len(self.ready_queue) == 0
        no_running_tasks = self._count_running_tasks() == 0
        all_tasks_handled = (self.completed_tasks + self.rejected_tasks) >= self.total_tasks
        return no_ready_tasks and no_running_tasks and all_tasks_handled

    def _get_resource_vector(self):
        """Flatten the resource matrix into a feature vector."""
        resource_features = []
        for server_index in range(self.num_servers):
            for vm_index in range(self.num_vms_per_server):
                resource_features.extend(self.resources[server_index][vm_index])
        return resource_features

    def _build_info(self, status, task=None, reward=0.0, action=None, reason=None):
        """Build the info dictionary returned by step()."""
        return {
            "status": status,
            "reason": reason,
            "action": action,
            "task": None if task is None else {
                "job_id": task.job_id,
                "task_index": task.task_index,
                "cpu_demand": task.cpu_demand,
                "ram_demand": task.ram_demand,
                "disk_demand": task.disk_demand,
                "runtime": task.runtime,
                "deadline": task.deadline,
            },
            "reward": reward,
            "completed_tasks": self.completed_tasks,
            "rejected_tasks": self.rejected_tasks,
            "running_tasks": self._count_running_tasks(),
            "ready_tasks": len(self.ready_queue),
            "current_time": self.current_time,
        }

    def get_state(self):
        """Return the current observation vector."""
        task = self._current_task()
        if task is None:
            task_features = [0.0] * 6
        else:
            wait_time = self._task_wait_time(task)
            task_features = [
                task.cpu_demand,
                task.ram_demand,
                task.disk_demand,
                task.runtime / self.max_runtime if self.max_runtime > 0 else 0.0,
                task.deadline / self.max_deadline if self.max_deadline > 0 else 0.0,
                wait_time / float(self.max_steps),
            ]

        summary_features = [
            self.current_time / float(self.max_steps),
            len(self.ready_queue) / float(max(1, self.total_tasks)),
            self._count_running_tasks() / float(max(1, self.total_tasks)),
        ]
        return np.array(summary_features + task_features + self._get_resource_vector(), dtype=np.float32)

    def get_valid_actions(self):
        """Return the actions that can currently schedule the ready task."""
        task = self._current_task()
        valid_actions = [self.wait_action]
        if task is None:
            return valid_actions

        for action_index, (server_index, vm_index) in enumerate(self.action_map):
            if self._can_place_task(task, server_index, vm_index):
                valid_actions.append(action_index)
        return valid_actions

    def reset(self, seed=None, options=None):
        """Reset the environment and return the initial observation."""
        super().reset(seed=seed)
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

        self.tasks = copy.deepcopy(self.job_templates)
        self.resources = self._build_server_resources()
        self.running_tasks = self._build_running_task_slots()
        self.ready_queue = deque()
        self.current_time = 0.0
        self.completed_tasks = 0
        self.rejected_tasks = 0
        self.steps_taken = 0
        self.terminated = False
        self.truncated = False

        for task in self.tasks:
            task.status = "pending"
            task.ready_time = None
            task.start_time = None
            task.finish_time = None
            task.assigned_location = None
            task.ready_time = self.current_time
            task.status = "ready"
            self.ready_queue.append(task)

        self._release_completed_tasks()
        return self.get_state(), self._build_info("reset")

    def step(self, action):
        """Advance the environment by one decision step."""
        if self.terminated:
            return self.get_state(), 0.0, True, self.truncated, self._build_info("terminated", reward=0.0, action=action)
        if self.truncated:
            return self.get_state(), 0.0, False, True, self._build_info("truncated", reward=0.0, action=action)

        # First, release any tasks that have completed by now to free up resources.
        self._release_completed_tasks()
        # Then, handle the current ready task using the chosen action and compute the reward.
        reward, status, task, reason = self._finalize_ready_task(action)
        self._advance_time()
        self.steps_taken += 1

        if self.steps_taken >= self.max_steps and not self._is_done():
            self.truncated = True

        self.terminated = self._is_done()
        info = self._build_info(status, task=task, reward=reward, action=action, reason=reason)
        return self.get_state(), reward, self.terminated, self.truncated, info

    def render(self):
        """Print a short text summary of the current state."""
        print(
            "time={0:.1f}, ready={1}, running={2}, completed={3}, rejected={4}".format(
                self.current_time,
                len(self.ready_queue),
                self._count_running_tasks(),
                self.completed_tasks,
                self.rejected_tasks,
            )
        )

    def close(self):
        """Release environment resources."""
        return None

In [ ]:
env = CloudSchedulingEnv(scale="small", workload_path="dataset.txt", num_tasks=None, num_servers=20, num_vms_per_server=2)
state, info = env.reset(seed=42)
print("state_size={0}, action_space_size={1}".format(env.state_size, env.action_space_size))
print("initial_ready_tasks={0}".format(info["ready_tasks"]))
for _ in range(50):
    action = random.choice(env.get_valid_actions())
    state, reward, done, truncated, info = env.step(action)
    print("reward={0:.4f}, status={1}, done={2}, truncated={3}".format(reward, info["status"], done, truncated))

## Part 2: Student Implementation Section

### 2.1. TODO: Complete the Scheduler

In this section, students will add their own reinforcement learning or heuristic scheduling solution.
TODO:
- Define a scheduler or agent for `CloudSchedulingEnv`.
- Train the agent or policy on the CloudSchedulingEnv workload.
- Record reward, completion rate, rejection rate, and runtime.
- Compare the learned policy against a simple baseline.

In [ ]:
# TODO: implement the student solution here.
# Suggested next steps:
# 1. Define a scheduler or RL agent.
# 2. Train it on CloudSchedulingEnv.
# 3. Evaluate reward, completion rate, rejection rate, and runtime.

---
# CONGRATULATIONS!

This Lab 06 notebook now provides a complete cloud scheduling setup for reinforcement learning studies:
- a Gymnasium-style `CloudSchedulingEnv` for resource provisioning and task scheduling
- a clear workload model, state design, action space, and reward structure
- dedicated student implementation placeholders for scheduling algorithms and evaluation
- a shared environment for comparing rule-based and learned scheduling policies

Technical takeaways:
- resource-aware scheduling depends on state quality, action constraints, and reward shaping
- a valid Gymnasium API makes the environment easier to test, train, and compare
- evaluation should consider reward, completion rate, rejection rate, and runtime behavior

## References

- Inspired by Shahid Mohammed Shaikbepari in Deep-Reinforcement-Learning-for-cloud repository
- Gymnasium documentation: https://gymnasium.farama.org/
- Stable-Baselines3 documentation: https://stable-baselines3.readthedocs.io/

---

## ADDITIONAL INFORMATION

**Author**: M.Sc. Phan Trung Phat - Department of Computer Networks and Communications, UIT

**Contact**: phatpt@uit.edu.vn

**Last Updated**: May, 2026